In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.metrics import classification_report
import numpy as np

# ------------------ LOAD DATA ------------------ #

df = pd.read_csv("sentence_level_dataset.csv")

# remove bad labels if any
df = df[df["label"].notna()]

print("Total samples:", len(df))
print(df["label"].value_counts())

# ------------------ TRAIN-VAL SPLIT ------------------ #

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# ------------------ LABEL ENCODING ------------------ #

labels = sorted(df["label"].unique())

label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)

# ------------------ CLASS WEIGHTS ------------------ #

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(list(label2id.values())),
    y=train_df["label_id"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class weights:", class_weights)

# ------------------ TOKENIZER ------------------ #

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# ------------------ DATASET ------------------ #

train_ds = Dataset.from_pandas(train_df[["sentence", "label_id"]])
val_ds = Dataset.from_pandas(val_df[["sentence", "label_id"]])

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])

# Rename the 'label_id' column to 'labels' as expected by the Trainer
train_ds = train_ds.rename_column("label_id", "labels")
val_ds = val_ds.rename_column("label_id", "labels")

# ------------------ MODEL ------------------ #

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# ------------------ CUSTOM TRAINER (CLASS WEIGHTS) ------------------ #

from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)

        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


# ------------------ TRAINING ARGS ------------------ #

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

# ------------------ TRAINER ------------------ #

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

# ------------------ TRAIN ------------------ #

trainer.train()

# ------------------ EVALUATION ------------------ #

preds = trainer.predict(val_ds)
y_pred = preds.predictions.argmax(axis=1)
y_true = preds.label_ids

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=labels))

# ------------------ SAVE MODEL ------------------ #

model.save_pretrained("bias_model")
tokenizer.save_pretrained("bias_model")

print("✅ Model saved as 'bias_model'")

Total samples: 11885
label
Performance Appraisal            2828
Expectation Bias                 1788
Disposition Bias                 1729
Omission Focus                   1543
Moral or Political Projection     881
Authenticity Bias                 841
Comparative Lens                  820
Representation Critique           752
Selective Emphasis Bias           703
Name: count, dtype: int64
Class weights: tensor([1.5698, 1.6104, 0.7639, 0.7388, 1.4985, 0.8554, 0.4670, 1.7549, 1.8798])


Map:   0%|          | 0/9508 [00:00<?, ? examples/s]

Map:   0%|          | 0/2377 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Classification Report:

                               precision    recall  f1-score   support

            Authenticity Bias       0.63      0.65      0.64       168
             Comparative Lens       0.70      0.65      0.68       164
             Disposition Bias       0.70      0.75      0.72       346
             Expectation Bias       0.65      0.59      0.62       358
Moral or Political Projection       0.70      0.71      0.71       176
               Omission Focus       0.69      0.52      0.59       308
        Performance Appraisal       0.82      0.82      0.82       566
      Representation Critique       0.61      0.59      0.60       150
      Selective Emphasis Bias       0.39      0.63      0.48       141

                     accuracy                           0.68      2377
                    macro avg       0.65      0.66      0.65      2377
                 weighted avg       0.69      0.68      0.68      2377



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved as 'bias_model'


In [ ]:
import shutil
shutil.make_archive("bias_model", 'zip', "bias_model")

from google.colab import files
files.download("bias_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("bias_model")
model = AutoModelForSequenceClassification.from_pretrained("bias_model")

def predict(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

    return model.config.id2label[pred.item()], conf.item()

print(predict("The acting was amazing but I expected more from the story"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

('Expectation Bias', 0.9628238677978516)


In [ ]:
predict('amazing acting')

('Performance Appraisal', 0.9567000269889832)